# Supplementary Figures S1-S5

Reproduces the supplementary eigenspectrum, parameter, convergence, timing, and theory panels from Shrinivas & Brenner (PNAS 2021).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import time

from jax_phase_separation.utils import (
    generate_chi_matrix, generate_initial_conditions, build_params,
    plot_volume_fractions, plot_phase_map, plot_partition_ratios,
)
from jax_phase_separation.solver import simulate, simulate_with_snapshots
from jax_phase_separation.free_energy import compute_jacobian, stability_analysis
from jax_phase_separation.analysis import analyse_snapshot
from jax_phase_separation.theory import (
    wigner_semicircle_pdf, n_phases_wigner, n_phases_tracy_widom,
)

print('JAX version:', jax.__version__)

---
## Figure S1: Eigenspectrum of the Jacobian

**(A)** Demixing instability (nu=0, sigma=4.8, N=20)  
**(B)** Condensation instability (nu=-6, sigma=1.2, N=20, beta=0.1)  
**(C)** Angle of marginal eigenvector with (1,1,...,1)

In [ ]:
N_COM_S1 = 20
N_REALIZATIONS = 200

# S1A: Demixing
sigma_demix = 4.8
beta_demix = N_COM_S1 / (N_COM_S1 + 1.0)
all_eigs_demix = []
angles_demix = []

for i in range(N_REALIZATIONS):
    key = jax.random.PRNGKey(i)
    chi = generate_chi_matrix(N_COM_S1, 0.0, sigma_demix, key)
    J = compute_jacobian(N_COM_S1, beta_demix, chi,
                         jnp.zeros(N_COM_S1), jnp.ones(N_COM_S1))
    w, v, _ = stability_analysis(J)
    # Normalise: (w - N/beta) / (sigma*sqrt(N))
    w_norm = (np.array(w) - N_COM_S1 / beta_demix) / (sigma_demix * np.sqrt(N_COM_S1))
    all_eigs_demix.extend(w_norm.tolist())
    # Angle of min eigenvector with (1,...,1)
    ones = np.ones(N_COM_S1) / np.sqrt(N_COM_S1)
    v_min = np.array(v[:, 0])  # smallest eigenvalue
    cos_a = np.clip(np.dot(ones, v_min), -1, 1)
    angles_demix.append(np.degrees(np.arccos(np.abs(cos_a))))

# S1B: Condensation
sigma_cond = 1.2
nu_cond = -6.0
beta_cond = 0.1
all_eigs_cond = []
angles_cond = []

for i in range(N_REALIZATIONS):
    key = jax.random.PRNGKey(i + 10000)
    chi = generate_chi_matrix(N_COM_S1, nu_cond, sigma_cond, key)
    J = compute_jacobian(N_COM_S1, beta_cond, chi,
                         jnp.zeros(N_COM_S1), jnp.ones(N_COM_S1))
    w, v, _ = stability_analysis(J)
    w_norm = (np.array(w) - N_COM_S1 / beta_cond) / (sigma_cond * np.sqrt(N_COM_S1))
    all_eigs_cond.extend(w_norm.tolist())
    ones = np.ones(N_COM_S1) / np.sqrt(N_COM_S1)
    v_min = np.array(v[:, 0])
    cos_a = np.clip(np.dot(ones, v_min), -1, 1)
    angles_cond.append(np.degrees(np.arccos(np.abs(cos_a))))

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

# S1A
R_demix = 2.0  # normalised semicircle radius
x_sc = np.linspace(-2.5, 2.5, 300)
ax1.hist(all_eigs_demix, bins=60, density=True, alpha=0.5, color='green', label='simulation')
ax1.plot(x_sc, wigner_semicircle_pdf(x_sc, R_demix), 'k-', lw=2, label='semicircle')
ax1.set_xlabel('$\\lambda_{norm}$', fontsize=12)
ax1.set_ylabel('$p(\\lambda_{norm})$', fontsize=12)
ax1.set_title('S1A: Demixing eigenspectrum', fontsize=12)
ax1.legend()

# S1B
ax2.hist(all_eigs_cond, bins=60, density=True, alpha=0.5, color='green', label='simulation')
ax2.plot(x_sc, wigner_semicircle_pdf(x_sc, R_demix), 'k-', lw=2, label='semicircle')
ax2.set_xlabel('$\\lambda_{norm}$', fontsize=12)
ax2.set_ylabel('$p(\\lambda_{norm})$', fontsize=12)
ax2.set_title('S1B: Condensation eigenspectrum', fontsize=12)
ax2.legend()

# S1C
ax3.hist(angles_demix, bins=30, alpha=0.5, color='purple', density=True, label='demixing')
ax3.hist(angles_cond, bins=30, alpha=0.5, color='goldenrod', density=True, label='condensation')
ax3.set_xlabel('$\\theta$ (degrees)', fontsize=12)
ax3.set_ylabel('$p(\\theta)$', fontsize=12)
ax3.set_title('S1C: Marginal eigenvector angle', fontsize=12)
ax3.legend()

plt.tight_layout()
plt.show()

---
## Figure S2: N=12, sigma=5.2 alternative parameter set

In [ ]:
N_COM_S2 = 12
SIGMA_S2 = 5.2
BETA_S2 = N_COM_S2 / (N_COM_S2 + 1.0)

key = jax.random.PRNGKey(77)
k1, k2 = jax.random.split(key)
chi_s2 = generate_chi_matrix(N_COM_S2, 0.0, SIGMA_S2, k1)
c0_s2 = generate_initial_conditions(N_COM_S2, 64, beta=BETA_S2, noise_strength=0.01, key=k2)
params_s2 = build_params(chi_s2, N_COM_S2, beta=BETA_S2)

print('Running S2 simulation...')
t0 = time.time()
c_final_s2 = simulate(c0_s2, params_s2, 64, 2_000_000, progress_bar=True)
print(f'Done in {time.time()-t0:.1f}s')

result_s2 = analyse_snapshot(np.array(c_final_s2))
print(f'Detected {result_s2["n_phases"]} phases')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Volume fractions (first 4 components)
for i in range(4):
    if i < 3:
        ax = axes[i] if i < 2 else None

# Left: show a few component maps
fig_vf, _ = plot_volume_fractions(c_final_s2, ncols=4, vmin=0, vmax=0.75)
fig_vf.suptitle(f'S2: Volume fractions (N={N_COM_S2}, sigma={SIGMA_S2})', fontsize=13, y=1.02)
plt.show()

# Phase map and partition ratios
fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
plot_phase_map(result_s2['labels'], result_s2['n_phases'], ax=ax1)
ax1.set_title(f'S2: Phase map ({result_s2["n_phases"]} phases)')
plot_partition_ratios(result_s2['partitions'], ax=ax2)
ax2.set_title('S2: Partition ratios')
plt.tight_layout()
plt.show()

---
## Figure S3: Convergence tests

**(A)** Mesh resolution (L=64 vs L=128)  
**(B)** Mobility model comparison  
**(C)** Extended simulation length

In [ ]:
N_COM_S3 = 16
SIGMA_S3 = 4.8
BETA_S3 = N_COM_S3 / (N_COM_S3 + 1.0)
N_REPS_S3 = 10
N_STEPS_SHORT = 2_000_000
SAVE_EVERY_S3 = 100_000


def run_convergence(n_grid, n_steps, mobility_flag=False, n_reps=N_REPS_S3):
    master = jax.random.PRNGKey(42)
    keys = jax.random.split(master, n_reps * 2).reshape(n_reps, 2, -1)
    all_nph = []
    for rep in range(n_reps):
        chi = generate_chi_matrix(N_COM_S3, 0.0, SIGMA_S3, keys[rep, 0])
        c0 = generate_initial_conditions(N_COM_S3, n_grid, beta=BETA_S3,
                                         noise_strength=0.01, key=keys[rep, 1])
        params = build_params(chi, N_COM_S3, beta=BETA_S3)
        _, snaps = simulate_with_snapshots(
            c0, params, n_grid, n_steps, save_every=SAVE_EVERY_S3,
            mobility_flag=mobility_flag, progress_bar=True,
        )
        nph_t = [analyse_snapshot(np.array(snaps[si]))['n_phases']
                 for si in range(snaps.shape[0])]
        all_nph.append(nph_t)
    return np.array(all_nph)


print('S3A: L=64...')
nph_64 = run_convergence(64, N_STEPS_SHORT)
print('S3A: L=128...')
nph_128 = run_convergence(128, N_STEPS_SHORT)
print('S3B: Mobility flag...')
nph_mob = run_convergence(64, N_STEPS_SHORT, mobility_flag=True)
print('Done.')

In [ ]:
t_axis = np.arange(1, nph_64.shape[1] + 1) * SAVE_EVERY_S3 * 5e-6

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# S3A: mesh
for data, label, color in [(nph_64, 'L=64', 'steelblue'), (nph_128, 'L=128', 'coral')]:
    m, s = data.mean(0), data.std(0)
    ax1.fill_between(t_axis, m - s, m + s, alpha=0.2, color=color)
    ax1.plot(t_axis, m, color=color, lw=2, label=label)
ax1.set_xscale('log')
ax1.set_xlabel('time'); ax1.set_ylabel('$N_{phases}$')
ax1.set_title('S3A: Mesh convergence'); ax1.legend()

# S3B: mobility
for data, label, color in [(nph_64, '$M_i=M\\phi_i$', 'steelblue'),
                            (nph_mob, '$M_i=M\\phi_i(1-\\phi_i)$', 'coral')]:
    m, s = data.mean(0), data.std(0)
    ax2.fill_between(t_axis, m - s, m + s, alpha=0.2, color=color)
    ax2.plot(t_axis, m, color=color, lw=2, label=label)
ax2.set_xscale('log')
ax2.set_xlabel('time'); ax2.set_ylabel('$N_{phases}$')
ax2.set_title('S3B: Mobility comparison'); ax2.legend()

plt.tight_layout()
plt.show()

---
## Figure S4: Eigenvector angles and phase timing

In [ ]:
# S4A: Angles between eigenvectors of the Jacobian
N_COM_S4 = 16
SIGMA_S4 = 4.8
BETA_S4 = N_COM_S4 / (N_COM_S4 + 1.0)

all_evec_angles = []
for i in range(200):
    key = jax.random.PRNGKey(i + 5000)
    chi = generate_chi_matrix(N_COM_S4, 0.0, SIGMA_S4, key)
    J = compute_jacobian(N_COM_S4, BETA_S4, chi,
                         jnp.zeros(N_COM_S4), jnp.ones(N_COM_S4))
    _, v, _ = stability_analysis(J)
    v_np = np.array(v)
    for a in range(v_np.shape[1]):
        for b in range(a + 1, v_np.shape[1]):
            cos_ab = np.clip(np.dot(v_np[:, a], v_np[:, b]), -1, 1)
            all_evec_angles.append(np.degrees(np.arccos(np.abs(cos_ab))))

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(all_evec_angles, bins=40, density=True, alpha=0.6, color='steelblue', label='pdf')
ax2 = ax.twinx()
sorted_a = np.sort(all_evec_angles)
ax2.plot(sorted_a, np.arange(1, len(sorted_a)+1)/len(sorted_a),
         color='darkorange', lw=2, label='cdf')
ax.set_xlabel('$\\theta$ (degrees)')
ax.set_ylabel('pdf')
ax2.set_ylabel('cdf', color='darkorange')
ax.set_title('S4A: Angles between Jacobian eigenvectors')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2)
plt.tight_layout()
plt.show()

---
## Figure S5: Theory with Tracy-Widom corrections

In [ ]:
# S5A: N_phases vs N for different sigma with Tracy-Widom corrections
sigma_vals = [3.0, 4.0, 5.0]
N_range = np.arange(4, 22)
N_cont = np.linspace(3, 22, 200)

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for si, sigma in enumerate(sigma_vals):
    # Wigner prediction
    ax.plot(N_cont, n_phases_wigner(N_cont, sigma), '-',
            color=colors[si], lw=2, label=f'Wigner $\\sigma$={sigma}')
    # Tracy-Widom correction
    ax.plot(N_cont, n_phases_tracy_widom(N_cont, sigma), '--',
            color=colors[si], lw=2, label=f'Tracy-Widom $\\sigma$={sigma}')

ax.set_xlabel('$N_{components}$', fontsize=12)
ax.set_ylabel('$N_{phases}$', fontsize=12)
ax.set_title('S5A: Theory predictions with Tracy-Widom corrections', fontsize=13)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# S5B: N_phases vs beta
N_COM_S5B = 16
SIGMA_S5B = 4.8
beta_range = np.linspace(0.3, 0.95, 100)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(beta_range, n_phases_wigner(N_COM_S5B, SIGMA_S5B, beta=beta_range),
        'b-', lw=2, label='Wigner theory')
ax.set_xlabel('$\\beta$ (total solute fraction)', fontsize=12)
ax.set_ylabel('$N_{phases}$', fontsize=12)
ax.set_title(f'S5B: Phases vs beta (N={N_COM_S5B}, sigma={SIGMA_S5B})', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# S5C-F: Extended theory predictions for alpha and sigma ensembles
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# S5C: alpha-ensemble, N_phases vs N for larger alpha range
N_cont = np.linspace(2, 50, 200)
for alpha in [0.6, 1.2, 1.8]:
    sigma = alpha * np.sqrt(N_cont)
    axes[0, 0].plot(N_cont, n_phases_wigner(N_cont, sigma),
                    lw=2, label=f'$\\alpha$={alpha}')
axes[0, 0].set_xlabel('$N_{components}$')
axes[0, 0].set_ylabel('$N_{phases}$')
axes[0, 0].set_title('S5C: $\\alpha$-ensemble theory')
axes[0, 0].legend()

# S5D: alpha-ensemble, N_phases vs alpha
alpha_cont = np.linspace(0.3, 3.0, 200)
for nc in [4, 12, 20]:
    sigma = alpha_cont * np.sqrt(nc)
    nph = n_phases_wigner(nc, sigma)
    axes[0, 1].plot(alpha_cont, nph, lw=2, label=f'N={nc}')
    axes[0, 1].axhline((nc + 1) / 2, ls=':', color='grey', lw=0.8)
axes[0, 1].set_xlabel('$\\alpha$')
axes[0, 1].set_ylabel('$N_{phases}$')
axes[0, 1].set_title('S5D: $\\alpha$-ensemble vs alpha')
axes[0, 1].legend()

# S5E: sigma-ensemble, N_phases vs N
N_cont = np.linspace(2, 50, 200)
for sigma in [2.0, 3.0, 4.0, 5.0]:
    axes[1, 0].plot(N_cont, n_phases_wigner(N_cont, sigma),
                    lw=2, label=f'$\\sigma$={sigma}')
axes[1, 0].set_xlabel('$N_{components}$')
axes[1, 0].set_ylabel('$N_{phases}$')
axes[1, 0].set_title('S5E: $\\sigma$-ensemble theory')
axes[1, 0].legend()

# S5F: sigma-ensemble, N_phases vs sigma
sigma_cont = np.linspace(0.5, 12, 200)
for nc in [4, 12, 20]:
    nph = n_phases_wigner(nc, sigma_cont)
    axes[1, 1].plot(sigma_cont, nph, lw=2, label=f'N={nc}')
    axes[1, 1].axhline((nc + 1) / 2, ls=':', color='grey', lw=0.8)
axes[1, 1].set_xlabel('$\\sigma$')
axes[1, 1].set_ylabel('$N_{phases}$')
axes[1, 1].set_title('S5F: $\\sigma$-ensemble vs sigma')
axes[1, 1].legend()

plt.tight_layout()
plt.show()